# NetCore TAC Desk: Why fixed ETAs miss

Run the cells in order from the workspace root or the `notebooks` folder. Load the synthetic tickets and check the SLA rule: a ticket breaches only if it takes longer than its own severity's ETA. A long-running ticket, by contrast, takes more than 72 hours at any severity.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.ticker import PercentFormatter

workspace_root = Path.cwd()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

tickets = pd.read_csv(workspace_root / "data" / "tac_tickets.csv", parse_dates=["created_at"])
fixed_etas = {"S1": 4, "S2": 24, "S3": 48, "S4": 72}
severity_order = list(fixed_etas)
severity_colors = dict(zip(severity_order, sns.color_palette("colorblind", 4), strict=True))
long_running_cutoff = 72
eta_hours = tickets["severity"].map(fixed_etas)
resolution_hours = tickets["resolution_hours"]
if eta_hours.isna().any() or resolution_hours.isna().any():
    raise ValueError("Every ticket needs a known severity and recorded resolution time.")

sla_breached = resolution_hours > eta_hours
if tickets["sla_breached"].isna().any() or not sla_breached.eq(tickets["sla_breached"]).all():
    raise ValueError("Recalculated SLA breaches do not match the stored flags.")

is_long_running = resolution_hours > long_running_cutoff
print(f"Tickets: {len(tickets):,} | SLA breach rate: {sla_breached.mean():.1%} | Over 72 hours: {is_long_running.sum():,}")

## Matplotlib basics: Monthly ticket volume

`plt.subplots()` creates a `Figure` (the canvas) and `Axes` (the plotting area). The line connects monthly ticket counts in UTC. Peaks show when intake was highest, which helps plan staffing; this chart does not show backlog.

In [ ]:
monthly_volume = tickets.groupby(pd.Grouper(key="created_at", freq="MS")).size()

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
ax.plot(monthly_volume.index, monthly_volume, marker="o", color="#0072B2")
ax.set(title="Monthly TAC Desk ticket intake", xlabel="Month created (UTC)", ylabel="Tickets created")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3, tz="UTC"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y", tz="UTC"))
ax.tick_params(axis="x", labelrotation=30)
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.25)
plt.show()

## Histogram: Actual resolution hours vs. fixed ETAs

The bars show how often different resolution times occur. The long tail shows that some cases take far longer than the fixed estimates. This chart mixes severities, so the count to the right of any one line is **not** that severity's breach count.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
sns.histplot(data=tickets, x="resolution_hours", bins=35, color="#9CA3AF", ax=ax)
for severity, hours in fixed_etas.items():
    ax.axvline(hours, color=severity_colors[severity], linestyle="--", label=f"{severity}: {hours} h")

ax.set(title="Resolution times and fixed ETAs", xlabel="Actual resolution time (hours)", ylabel="Tickets per bin")
ax.legend(title="Fixed ETA", loc="upper right")
plt.show()

## Boxplot: Resolution time by severity

Each box contains the middle half of tickets at that severity, with a line at the median. Compare the spread to the ETA diamond. Tickets at the same severity resolve at different speeds, so one fixed estimate can be too short for some and too long for others.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
sns.boxplot(data=tickets, x="severity", y="resolution_hours", order=severity_order, color="#A6CEE3", ax=ax)
ax.scatter(range(len(severity_order)), list(fixed_etas.values()), marker="D", s=55,
           color="#B33C12", edgecolor="white", zorder=4, label="Fixed ETA")

ax.set(title="Actual resolution time by severity", xlabel="Severity", ylabel="Actual resolution time (hours)")
ax.legend(loc="upper left")
plt.show()

## Bar chart: SLA breach rate by region

Each bar divides the number of tickets that exceeded their own ETA by all tickets in that region. Compare the rates to decide where to investigate, but check ticket volume and severity mix before attributing differences to regional performance.

In [ ]:
regional_rates = (
    tickets.assign(breached=sla_breached)
    .groupby("region", as_index=False)
    .agg(breach_rate=("breached", "mean"), ticket_count=("ticket_id", "size"))
)

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
sns.barplot(data=regional_rates, x="region", y="breach_rate", errorbar=None, color="#0072B2", ax=ax)
ax.bar_label(ax.containers[0], labels=[f"{rate:.1%}" for rate in regional_rates["breach_rate"]], padding=4)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(ylim=(0, 1), title="SLA breach rate by region", xlabel="Support region", ylabel="Tickets that breached SLA (%)")
plt.show()

## Scatter plot: Which tickets took over 72 hours?

Each point is a ticket. Color shows severity; an X above the dashed line marks a long-running case. Look within each color to see how device count relates to resolution time. Cases can run long even with few devices, and the chart does not show that device count caused the delay.

In [ ]:
duration_group = is_long_running.map({False: "72 h or less", True: "Over 72 h"})
scatter_data = tickets.assign(duration_group=duration_group)

fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
sns.scatterplot(
    data=scatter_data, x="devices_affected", y="resolution_hours", hue="severity",
    hue_order=severity_order, palette=severity_colors, style="duration_group",
    style_order=["72 h or less", "Over 72 h"], markers={"72 h or less": "o", "Over 72 h": "X"},
    s=30, alpha=0.65, linewidth=0, ax=ax,
)
ax.axhline(long_running_cutoff, color="#333333", linestyle="--", label="72 h cutoff")
ax.set(title="Devices affected and long-running tickets", xlabel="Devices affected (count)",
       ylabel="Actual resolution time (hours)")
ax.legend(loc="upper right")
plt.show()

## Heatmap: Correlations between numeric fields

Pearson correlation measures linear association from -1 to +1. Read the resolution-hours row to spot relationships worth investigating, not causes. Use only original numeric columns (no flags or derived metrics); each pair uses rows with both values present. Escalation count can be useful for retrospective analysis without being available at triage.

In [ ]:
numeric_tickets = tickets.select_dtypes(include="number").rename(columns={
    "devices_affected": "Devices affected",
    "software_version_months": "Software age (months)",
    "engineer_experience_years": "Engineer experience (years)",
    "first_response_minutes": "First response (minutes)",
    "escalation_count": "Escalation count",
    "resolution_hours": "Resolution (hours)",
})
correlations = numeric_tickets.corr(method="pearson")

fig, ax = plt.subplots(figsize=(10, 7), layout="constrained")
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, center=0,
            cbar_kws={"label": "Pearson r"}, ax=ax)
ax.set(title="Correlation of original numeric ticket fields", xlabel="Numeric ticket field", ylabel="Numeric ticket field")
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)
plt.show()